# Notebook 04 — Validação e Tratamento de Erros

Nos notebooks anteriores, construí um sistema de controle de aulas que cadastra
alunos, calcula valores e salva os dados em JSON.

Mas o sistema ainda é **frágil**: se o usuário digitar uma letra onde eu espero um
número, ou tentar abrir um arquivo que não existe, o programa quebra.

Neste notebook vou aprender a deixar o sistema **robusto**, usando:

- `try` / `except` para tratar erros;
- validação de dados antes de aceitá-los;
- funções de entrada que insistem até o usuário digitar algo válido;
- `raise` para sinalizar dados inválidos.

## 1. O que acontece quando algo dá errado

Quando o Python encontra um erro em tempo de execução, ele lança uma **exceção**
e para o programa. Veja um exemplo clássico: tentar transformar um texto em número.

In [ ]:
# Isto gera um erro (ValueError) e interrompe a execução:
# valor = float("cento e trinta")

# Por isso, em vez de deixar quebrar, vamos "capturar" o erro.
try:
    valor = float("cento e trinta")
except ValueError:
    print("Não consegui converter esse texto em número.")
    valor = 0.0

print("valor =", valor)

## 2. A estrutura try / except

- O bloco `try` contém o código que **pode** dar erro.
- O bloco `except` é executado **somente** se o erro acontecer.

Posso até capturar a mensagem original do erro com `as erro`:

In [ ]:
try:
    idade = int("vinte")
except ValueError as erro:
    print("Deu erro:", erro)

## 3. Função de entrada que valida números

Em vez de confiar que o usuário vai digitar certo, crio uma função que **repete a
pergunta** até receber um número válido. Uso `while True` + `try/except`.

In [ ]:
def ler_float(mensagem):
    while True:
        try:
            # aceita vírgula ou ponto como separador decimal
            valor = float(input(mensagem).replace(",", "."))
        except ValueError:
            print("  -> Digite um número válido (ex.: 130 ou 130.50).")
            continue
        if valor <= 0:
            print("  -> O valor precisa ser maior que zero.")
            continue
        return valor

In [ ]:
def ler_int(mensagem):
    while True:
        try:
            valor = int(input(mensagem))
        except ValueError:
            print("  -> Digite um número inteiro (ex.: 2).")
            continue
        if valor <= 0:
            print("  -> O número precisa ser maior que zero.")
            continue
        return valor

## 4. Validando texto

Nome, série e disciplina não podem ficar vazios. Crio uma função que insiste
enquanto o usuário só apertar Enter ou digitar espaços.

In [ ]:
def ler_texto(mensagem):
    while True:
        valor = input(mensagem).strip()
        if valor:
            return valor
        print("  -> Esse campo não pode ficar vazio. Tente de novo.")

## 5. Sinalizando erros com `raise`

Às vezes eu mesmo quero **gerar** um erro quando um dado não faz sentido — por
exemplo, um valor de aula negativo. Para isso uso `raise ValueError(...)`.

Assim quem chamar a função fica responsável por tratar o problema.

In [ ]:
def validar_valor_aula(valor):
    valor = float(valor)
    if valor <= 0:
        raise ValueError("O valor da aula precisa ser maior que zero.")
    return valor


# Testando os dois caminhos:
print(validar_valor_aula(130))

try:
    validar_valor_aula(-50)
except ValueError as erro:
    print("Erro capturado:", erro)

## 6. Cadastro de aluno com validação

Agora junto tudo: uma função de cadastro que valida os dados e **não** deixa
entrar um aluno inválido na lista.

In [ ]:
def cadastrar_aluno(lista_alunos, nome, serie, disciplina, valor_aula, aulas_por_semana):
    if not nome.strip():
        raise ValueError("O nome não pode ficar vazio.")
    if float(valor_aula) <= 0:
        raise ValueError("O valor da aula precisa ser maior que zero.")
    if int(aulas_por_semana) <= 0:
        raise ValueError("As aulas por semana precisam ser maior que zero.")

    novo_aluno = {
        "nome": nome.strip(),
        "serie": serie.strip(),
        "disciplina": disciplina.strip(),
        "valor_aula": float(valor_aula),
        "aulas_por_semana": int(aulas_por_semana),
    }
    lista_alunos.append(novo_aluno)
    print("Aluno cadastrado com sucesso:", novo_aluno["nome"])
    return novo_aluno

In [ ]:
alunos = []

# cadastro válido
cadastrar_aluno(alunos, "Ana", "9º ano", "Matemática", 130.0, 2)

# cadastro inválido (valor negativo) — tratado sem quebrar o programa
try:
    cadastrar_aluno(alunos, "Erro", "9º ano", "Matemática", -10, 2)
except ValueError as erro:
    print("Não cadastrei:", erro)

print("Total de alunos válidos:", len(alunos))

## 7. Tratando erros ao carregar arquivos

Dois problemas comuns ao ler um JSON:

- o arquivo **não existe** → `FileNotFoundError`;
- o arquivo existe mas está **corrompido** → `json.JSONDecodeError`.

A função abaixo trata os dois casos e devolve uma lista vazia em vez de quebrar.

In [ ]:
import json


def carregar_alunos_json(nome_arquivo):
    try:
        with open(nome_arquivo, "r", encoding="utf-8") as arquivo:
            return json.load(arquivo)
    except FileNotFoundError:
        print(f"Arquivo '{nome_arquivo}' não encontrado. Começando com lista vazia.")
        return []
    except json.JSONDecodeError:
        print(f"Arquivo '{nome_arquivo}' está corrompido. Começando com lista vazia.")
        return []


# arquivo que existe
dados = carregar_alunos_json("alunos.json")
print("Alunos carregados:", len(dados))

# arquivo que não existe
vazio = carregar_alunos_json("nao_existe.json")
print("Resultado:", vazio)

## Conclusão do Notebook 04

Neste notebook deixei o sistema muito mais seguro. Aprendi a:

- entender o que é uma **exceção**;
- usar `try` / `except` para não deixar o programa quebrar;
- capturar a mensagem do erro com `as erro`;
- criar funções de entrada que **insistem** até receber dados válidos
  (`ler_texto`, `ler_int`, `ler_float`);
- usar `raise` para sinalizar dados inválidos;
- validar dados antes de cadastrar;
- tratar `FileNotFoundError` e `JSONDecodeError` ao carregar arquivos.

### Próximo passo

No Notebook 05 vou reorganizar todo esse código usando **Orientação a Objetos**,
transformando o aluno e o próprio sistema em **classes**. Isso vai juntar dados e
comportamento em um só lugar e deixar o projeto pronto para virar um módulo de
verdade (`sistema_alunos.py`).